In [ ]:
import socket
import re
import subprocess
import random as rd

Note: Use hashpump

In [ ]:
def hashpump(signature_hex: str, data_pt: str, additional_data: str, key_len: int=256) -> dict:
    result = subprocess.run(["./HashPump-partialhash/hashpump", "-s", signature_hex, "-d", data_pt, "-a", additional_data, "-k", str(key_len)], stdout=subprocess.PIPE)
    result = result.stdout.decode()
    lines = re.split('\n|\r', result)

    data = dict()
    data["predicted_sig"] = re.search("[0-9a-fA-F]+", lines[2].split(':')[1]).group()
    data["appended"] = lines[3].strip()

    return data
    


hashpump("844c5b6a446eddf2b3d865943aa37bd1d9b520cf1fe91ee0ed2cb3e1c3ea68cb011cdda7ca9933ba111f68af862f27a98afaf1bcdf24bd76f679163a62faf9ba",
         "24",
         ", 1",
         256)

In [ ]:
temp = hashpump("3a87561b0bc11c67e211d51af3be9002d582a1a86d1c84850aebbde0ca8c4c38541507eb0b14380ba5fdf4d1b7343a013dbaad1ebe4f3071abf8a6dc6d4b3bdd",
         "162, -111",
         ", 1, 2, 3, 4")
print(temp["appended"])
print(temp["predicted_sig"])

In [ ]:
HOST="lonely-island.picoctf.net"
PORT=64069

attempt = 1

# Pass escaped string and hexstring
def query(s, vector_str: str, hash_hexstring: str) -> int:
    vector_bytestring = vector_str.encode()
    hash_bytestring = hash_hexstring.encode()


    # Send forged vector
    s.sendall(vector_bytestring + b'\n')

    # Receive until I am prompted for salted hash
    prompt = b""
    while b"Enter its salted hash: " not in prompt:
        prompt += s.recv(4096)

    # Send forged sig
    s.sendall(hash_bytestring + b'\n')

    # Receive until the next prompt for vector
    prompt = b""
    while b"Enter your vector: " not in prompt:
        if b"Untrusted" in prompt:
            raise RuntimeError("Server rejection of forged signature")
        prompt += s.recv(4096)
    dot_product = int(re.search("[0-9]+", prompt.decode()).group())
    return dot_product


    # Need at least one trusted vector that is of length 32
while True:
    #print(f"Request #{n_request}")
    #n_request += 1
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT))
        banner = b""
        while b"Enter your vector: " not in banner:
            banner += s.recv(8192)

        banner = banner.decode().split('\n')
        # Extract IV
        iv_str = banner[1]
        iv = bytes.fromhex(re.search("[0-9A-Fa-f]+", iv_str).group())
        # Extract CT
        ct_str = banner[2]
        ct = bytes.fromhex(re.search("[0-9A-Fa-f]{2,}", ct_str).group())
        # Extract trusted vectors
        trusted_vector_str = banner[6:-2]
        trusted_vectors = []
        #print(iv)
        #print(ct)
        for trusted_vec in trusted_vector_str:
            trusted_vectors.append(eval(trusted_vec))
        #print(trusted_vectors)

        length_one_trusted_vectors = list(filter(lambda entry: len(entry[0]) == 1, trusted_vectors))
        if length_one_trusted_vectors == []:
            continue

        data = []
        # Initial data point guaranteed to be length 1.
        # data = list of (v, salted_hash, dot_product)
        v, salted_hash = length_one_trusted_vectors[0]
        dot_product = query(s, str(v), salted_hash)         # This is guaranteed to succeed, as this is literally one of the given trusted vectors.

        data = [(v, salted_hash, dot_product)]

        while len(data) != 32:
            incr_len = len(data)        # The number of elements to add in the current iteration. If len(data) == 1, then incr_len = 2. If len(data) == 2, then need to add 2 more entries to the vector.
            v, salted_hash, _ = data[0]

            additional_coeffs = rd.choices(range(1,100), k=incr_len)
            additional_coeffs_str = ', ' + ", ".join(map(str, additional_coeffs))

            forged = hashpump(salted_hash, str(v)[1:-1], additional_coeffs_str, 256)
            dot_product = query(s, '[' + forged["appended"] + ']', forged["predicted_sig"])

            data.append((v + additional_coeffs, forged["predicted_sig"], dot_product))
    
    break

            


In [ ]:
import pickle as pkl
with open("data.pkl", "wb") as f:
    pkl.dump((iv, ct, data), f)
    #iv, ct, data


In [1]:
import pickle as pkl
with open("data.pkl", "rb") as f:
    iv2, ct2, data2 = pkl.load(f)
print(iv2)
print(ct2)
print(data2)

b'\x1c\x92\x1f\xd6\xe3\x17xi\xf3\xbfsQ\xf3\xf6\xe9-'
b"\xa9+\xd6M\x14\xfa\xd2^\xf4+\xee\xa8\xdc\xdb\xef\xac\xd7\xd1A\x03\xde\xe2.'\xde0Z\x19^\xf6\xb1\xd0\xf8$l\x81a\x9a\xc5\xb3jDy\x1e\x9b\x10\xf4\xbc"
[([2], '0fc337afa50d28710aee45ccab515df55b17bd03f2f84eba99d2969d4e4b953e6a3c27c646228228f336dda2d72130f89842c144b340c8e6c0c6ea0b01c89a6d', 238), ([2, 95], 'a957d96a03a517f1e50bbe675d21bc106cc9bf63346fa50ef3377187e528677c4a866bf13f31b1d1b57bf668964d7bdffe9bb0442cf5708bc42687a9054d1b69', 4513), ([2, 54, 35], 'e478912f8f5d28de8f9078cdb8b52e37b83955ac427dcf397c6222df572254ad10856b1d8d0a2d4779cc4bc01d00c2819bd7aadc55a44e3606b23dfdee663699', 6483), ([2, 52, 74, 98], 'c5bd219207a0f27cf0bfff93c2f5ce9ee310e3bea85978e532da387fa2dbe463b4f1545105e1e83621c654458571e3a9ae04de6b7140cd26d44dccccd9048741', 13192), ([2, 77, 86, 98, 46], '248789260aa531c5653f032e8cf315b514e67e2229c30789585536f4e4503ed37d143fd5bc9bcf2a159be97be23d6f05ed15143c636ecef0af46e9db13d177a1', 25515), ([2, 77, 95, 96, 94, 73], '53483

In [5]:
for v, _, d in data2:
    print(f"v: {v}, d: {d}")

v: [2], d: 238
v: [2, 95], d: 4513
v: [2, 54, 35], d: 6483
v: [2, 52, 74, 98], d: 13192
v: [2, 77, 86, 98, 46], d: 25515
v: [2, 77, 95, 96, 94, 73], d: 37056
v: [2, 20, 76, 76, 56, 96, 15], d: 26102
v: [2, 83, 18, 55, 84, 25, 31, 82], d: 39749
v: [2, 80, 1, 69, 79, 70, 83, 75, 13], d: 45817
v: [2, 81, 66, 90, 82, 12, 52, 59, 15, 95], d: 60488
v: [2, 47, 34, 47, 26, 46, 99, 59, 56, 45, 22], d: 49575
v: [2, 74, 12, 2, 79, 95, 44, 13, 78, 69, 47, 54], d: 60227
v: [2, 79, 67, 12, 68, 18, 57, 34, 26, 7, 62, 75, 13], d: 61351
v: [2, 7, 86, 99, 34, 56, 54, 48, 2, 16, 64, 46, 74, 80], d: 77011
v: [2, 68, 36, 39, 97, 93, 56, 89, 65, 30, 26, 56, 19, 46, 47], d: 81489
v: [2, 93, 53, 70, 78, 99, 20, 3, 73, 46, 15, 41, 95, 3, 13, 70], d: 83968
v: [2, 48, 84, 75, 9, 40, 2, 75, 28, 24, 94, 80, 36, 2, 88, 24, 88], d: 78301
v: [2, 22, 17, 94, 61, 85, 16, 4, 82, 48, 92, 41, 35, 12, 32, 92, 46, 73], d: 90835
v: [2, 40, 76, 2, 91, 64, 83, 10, 17, 80, 89, 71, 22, 27, 22, 56, 32, 53, 44], d: 111772
v: [2, 4

In [7]:
array = [v + [0 for _ in range(32 - len(v))] for v, _, _ in data2]
A = matrix(ZZ, 32, 32, array)
v = vector(ZZ, [d for _, _, d in data2])
print(A)
print(v)

[ 2  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 95  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 54 35  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 52 74 98  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 77 86 98 46  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 77 95 96 94 73  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 20 76 76 56 96 15  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 83 18 55 84 25 31 82  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 80  1 69 79 70 83 75 13  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 81 66 90 82 12 52 59 15 95  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 2 47 34 47 26 46 9

In [9]:
x = A.solve_right(v)
print(x)

(119, 45, 109, 26, 215, 4, 152, 116, 115, 136, 94, 129, 246, 133, 25, 85, 167, 172, 197, 191, 86, 38, 226, 43, 254, 154, 124, 179, 40, 22, 122, 187)


In [12]:
key = bytes(list(x))

In [ ]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad
cipher = AES.new(key, AES.MODE_CBC, iv2)
plaintext = unpad(cipher.decrypt(ct2), AES.block_size)

# Redacted
#print(plaintext)